In [68]:
# Retrain the VGG16 model (from hw3)
import argparse
import os
import time
import shutil

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torch.backends.cudnn as cudnn

    

import torchvision
import torchvision.transforms as transforms

from models import *   # bring everything in the folder models

global best_prec
use_gpu = torch.cuda.is_available()
print('=> Building model...')
    
    
batch_size = 128

model_name = "cifar10"
#model = cifar10()
model = VGG16()
# model = resnet20_cifar()

        

normalize = transforms.Normalize(mean=[0.491, 0.482, 0.447], std=[0.247, 0.243, 0.262])


train_dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        normalize,
    ]))
trainloader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)


test_dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=False,
    download=True,
    transform=transforms.Compose([
        transforms.ToTensor(),
        normalize,
    ]))

testloader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)


print_freq = 100 # every 100 batches, accuracy printed. Here, each batch includes "batch_size" data points
# CIFAR10 has 50,000 training data, and 10,000 validation data.

def train(trainloader, model, criterion, optimizer, epoch):
    batch_time = AverageMeter()   ## at the begining of each epoch, this should be reset
    data_time = AverageMeter()
    losses = AverageMeter()
    top1 = AverageMeter()

    model.train()

    end = time.time()  # measure current time
    
    for i, (input, target) in enumerate(trainloader):
        # measure data loading time
        data_time.update(time.time() - end)  # data loading time

        input, target = input.cuda(), target.cuda()

        # compute output
        output = model(input)
        loss = criterion(output, target)

        # measure accuracy and record loss
        prec = accuracy(output, target)[0]
        losses.update(loss.item(), input.size(0))
        top1.update(prec.item(), input.size(0))

        # compute gradient and do SGD step
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # measure elapsed time
        batch_time.update(time.time() - end) # time spent to process one batch
        end = time.time()


        if i % print_freq == 0:
            print('Epoch: [{0}][{1}/{2}]\t'
                  'Time {batch_time.val:.3f} ({batch_time.avg:.3f})\t'
                  'Data {data_time.val:.3f} ({data_time.avg:.3f})\t'
                  'Loss {loss.val:.4f} ({loss.avg:.4f})\t'
                  'Prec {top1.val:.3f}% ({top1.avg:.3f}%)'.format(
                   epoch, i, len(trainloader), batch_time=batch_time,
                   data_time=data_time, loss=losses, top1=top1))

            

def validate(val_loader, model, criterion ):
    batch_time = AverageMeter()
    losses = AverageMeter()
    top1 = AverageMeter()

    # switch to evaluate mode
    model.eval()

    end = time.time()
    with torch.no_grad():
        for i, (input, target) in enumerate(val_loader):
         
            input, target = input.cuda(), target.cuda()

            # compute output
            output = model(input)
            loss = criterion(output, target)

            # measure accuracy and record loss
            prec = accuracy(output, target)[0]
            losses.update(loss.item(), input.size(0))
            top1.update(prec.item(), input.size(0))

            # measure elapsed time
            batch_time.update(time.time() - end)
            end = time.time()

            if i % print_freq == 0:  # This line shows how frequently print out the status. e.g., i%5 => every 5 batch, prints out
                print('Test: [{0}/{1}]\t'
                  'Time {batch_time.val:.3f} ({batch_time.avg:.3f})\t'
                  'Loss {loss.val:.4f} ({loss.avg:.4f})\t'
                  'Prec {top1.val:.3f}% ({top1.avg:.3f}%)'.format(
                   i, len(val_loader), batch_time=batch_time, loss=losses,
                   top1=top1))

    print(' * Prec {top1.avg:.3f}% '.format(top1=top1))
    return top1.avg


def accuracy(output, target, topk=(1,5)):
    """Computes the precision@k for the specified values of k"""
    maxk = max(topk) # 5
    batch_size = target.size(0) # 128

    _, pred = output.topk(maxk, 1, True, True) # topk(k, dim=None, largest=True, sorted=True)
                                    # will output (max value, its index)
    pred = pred.t()               # transpose
    correct = pred.eq(target.view(1, -1).expand_as(pred))   # "-1": calculate automatically

    res = []
    for k in topk: # 1, 5
        correct_k = correct[:k].view(-1).float().sum(0)  # view(-1): make a flattened 1D tensor
        res.append(correct_k.mul_(100.0 / batch_size))   # correct: size of [maxk, batch_size]
    return res


class AverageMeter(object):
    """Computes and stores the average and current value"""
    def __init__(self):
        self.reset()

    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0

    def update(self, val, n=1):
        self.val = val
        self.sum += val * n    ## n is impact factor
        self.count += n
        self.avg = self.sum / self.count

        
def save_checkpoint(state, is_best, fdir):
    filepath = os.path.join(fdir, 'checkpoint.pth')
    torch.save(state, filepath)
    if is_best:
        shutil.copyfile(filepath, os.path.join(fdir, 'model_best.pth.tar'))


def adjust_learning_rate(optimizer, epoch):
    """For resnet, the lr starts from 0.1, and is divided by 10 at 80 and 120 epochs"""
    adjust_list = [150, 225]
    if epoch in adjust_list:
        for param_group in optimizer.param_groups:
            param_group['lr'] = param_group['lr'] * 0.1        

#model = nn.DataParallel(model).cuda()
#all_params = checkpoint['state_dict']
#model.load_state_dict(all_params, strict=False)
#criterion = nn.CrossEntropyLoss().cuda()
#validate(testloader, model, criterion)

=> Building model...
Files already downloaded and verified
Files already downloaded and verified


In [69]:
def accuracy(output, target, topk=(1,5)):
    """Computes the precision@k for the specified values of k"""
    maxk = max(topk) # 5
    batch_size = target.size(0) # 128

    _, pred = output.topk(maxk, 1, True, True) # topk(k, dim=None, largest=True, sorted=True)
                                    # will output (max value, its index)
    pred = pred.t()               # transpose
    correct = pred.eq(target.reshape(1, -1).expand_as(pred))   # "-1": calculate automatically

    res = []
    for k in topk: # 1, 5
        correct_k = correct[:k].reshape(-1).float().sum(0)  # view(-1): make a flattened 1D tensor
        res.append(correct_k.mul_(100.0 / batch_size))   # correct: size of [maxk, batch_size]
    return res

In [70]:
model = VGG16()


# This cell is from the website

lr = 0.1



weight_decay = 5e-4
epochs = 60
best_prec = 0

model = model.cuda()
criterion = nn.CrossEntropyLoss().cuda()
optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=weight_decay)
scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=[30, 45], gamma=0.1)

# weight decay: for regularization to prevent overfitting


if not os.path.exists('result'):
    os.makedirs('result')
    
fdir = 'result/'+str(model_name)

if not os.path.exists(fdir):
    os.makedirs(fdir)
        

for epoch in range(0, epochs):
    adjust_learning_rate(optimizer, epoch)

    train(trainloader, model, criterion, optimizer, epoch)
    
    # evaluate on test set
    print("Validation starts")
    prec = validate(testloader, model, criterion)

    scheduler.step()

    # remember best precision and save checkpoint
    is_best = prec > best_prec
    best_prec = max(prec,best_prec)
    print('best acc: {:1f}'.format(best_prec))
    save_checkpoint({
        'epoch': epoch + 1,
        'state_dict': model.state_dict(),
        'best_prec': best_prec,
        'optimizer': optimizer.state_dict(),
    }, is_best, fdir)
    

Epoch: [0][0/391]	Time 18.904 (18.904)	Data 18.539 (18.539)	Loss 2.5004 (2.5004)	Prec 13.281% (13.281%)
Epoch: [0][100/391]	Time 0.055 (0.243)	Data 0.001 (0.185)	Loss 2.3241 (3.1464)	Prec 10.938% (10.837%)
Epoch: [0][200/391]	Time 0.065 (0.151)	Data 0.002 (0.094)	Loss 2.2864 (2.7537)	Prec 14.844% (11.000%)
Epoch: [0][300/391]	Time 0.068 (0.123)	Data 0.002 (0.063)	Loss 2.2721 (2.5993)	Prec 10.938% (11.174%)
Validation starts
Test: [0/79]	Time 15.819 (15.819)	Loss 2.3021 (2.3021)	Prec 10.156% (10.156%)
 * Prec 12.420% 
best acc: 12.420000
Epoch: [1][0/391]	Time 15.680 (15.680)	Data 15.374 (15.374)	Loss 2.3528 (2.3528)	Prec 10.938% (10.938%)
Epoch: [1][100/391]	Time 0.055 (0.212)	Data 0.001 (0.154)	Loss 2.2479 (2.2706)	Prec 15.625% (12.082%)
Epoch: [1][200/391]	Time 0.065 (0.138)	Data 0.001 (0.078)	Loss 2.1993 (2.2573)	Prec 21.094% (13.192%)
Epoch: [1][300/391]	Time 0.064 (0.114)	Data 0.001 (0.052)	Loss 2.0863 (2.2252)	Prec 20.312% (14.672%)
Validation starts
Test: [0/79]	Time 14.557 (14.

In [71]:
# Get the model accuracy
criterion = nn.CrossEntropyLoss().cuda()

model.eval()
model.cuda()

prec = validate(testloader, model, criterion)

Test: [0/79]	Time 22.728 (22.728)	Loss 0.1696 (0.1696)	Prec 95.312% (95.312%)
 * Prec 92.170% 


In [72]:
# Get absolute sum of the 1st conv layer's weights
conv1 = model.features[0]
tot_weight = conv1.weight.abs().sum()
print(tot_weight)

tensor(111.2243, device='cuda:0', grad_fn=<SumBackward0>)


In [75]:
# Save the original VGG16 model
torch.save(model.state_dict(), "result/vgg16/vgg16.pth")

In [76]:
device = torch.device("cuda" if use_gpu else "cpu") 
# Using [Code11] as reference
n_epochs = 20
# Model has already been partially trained


for epoch in range(n_epochs):
    # monitor training loss
    train_loss = 0.0
    train_loss1 = 0.0
    train_loss2 = 0.0
    
    ###################
    # train the model #
    ###################
    for data, target in trainloader:
        data, target = data.to(device), target.to(device) # loading to GPU
        # clear the gradients of all optimized variables
        optimizer.zero_grad()
        # forward pass: compute predicted outputs by passing inputs to the model
        output = model(data)
        # calculate the loss
        
        loss1 = criterion(output, target)
        loss2 = model.features[0].weight.abs().sum()

        loss = loss1 + loss2
        
        loss.backward()
        optimizer.step()
        train_loss1 += loss1.item()*data.size(0) # as loss is tensor, .item() needed to get the value
        train_loss2 += loss2.item()*data.size(0) # as loss is tensor, .item() needed to get the value
        train_loss  += loss.item()*data.size(0) # as loss is tensor, .item() needed to get the value
        
    # print training statistics 
    # calculate average loss over an epoch
    train_loss1 = train_loss1/len(trainloader.dataset)
    train_loss2 = train_loss2/len(trainloader.dataset)
    train_loss = train_loss/len(trainloader.dataset)

    print('Epoch: {} \tTraining Loss1: {:.6f}'.format(epoch+1, train_loss1))
    print('Epoch: {} \tTraining Loss2: {:.6f}'.format(epoch+1, train_loss2))
    print('Epoch: {} \tTraining Loss: {:.6f}'.format(epoch+1, train_loss))
  

Epoch: 1 	Training Loss1: 0.606290
Epoch: 1 	Training Loss2: 9.148032
Epoch: 1 	Training Loss: 9.754322
Epoch: 2 	Training Loss1: 0.453417
Epoch: 2 	Training Loss2: 2.255413
Epoch: 2 	Training Loss: 2.708830
Epoch: 3 	Training Loss1: 0.380581
Epoch: 3 	Training Loss2: 2.216954
Epoch: 3 	Training Loss: 2.597535
Epoch: 4 	Training Loss1: 0.342659
Epoch: 4 	Training Loss2: 2.180033
Epoch: 4 	Training Loss: 2.522693
Epoch: 5 	Training Loss1: 0.327900
Epoch: 5 	Training Loss2: 2.175623
Epoch: 5 	Training Loss: 2.503524
Epoch: 6 	Training Loss1: 0.312901
Epoch: 6 	Training Loss2: 2.149406
Epoch: 6 	Training Loss: 2.462307
Epoch: 7 	Training Loss1: 0.306009
Epoch: 7 	Training Loss2: 2.159151
Epoch: 7 	Training Loss: 2.465160
Epoch: 8 	Training Loss1: 0.283025
Epoch: 8 	Training Loss2: 2.145398
Epoch: 8 	Training Loss: 2.428423
Epoch: 9 	Training Loss1: 0.279297
Epoch: 9 	Training Loss2: 2.141798
Epoch: 9 	Training Loss: 2.421095
Epoch: 10 	Training Loss1: 0.280023
Epoch: 10 	Training Loss2: 2

In [79]:
# Evaluate model after training w/ new loss function
criterion = nn.CrossEntropyLoss().cuda()

model.eval()
model.cuda()

prec = validate(testloader, model, criterion)

Test: [0/79]	Time 46.888 (46.888)	Loss 0.4213 (0.4213)	Prec 87.500% (87.500%)
 * Prec 86.950% 


In [78]:
# Get weight abs sum
print(model.features[0].weight.abs().sum())

tensor(2.1997, device='cuda:0', grad_fn=<SumBackward0>)


In [95]:
checkpoint = torch.load("result/vgg16/vgg16.pth")

# From hw4 solution
def load_original_model():
    model = VGG16()
    model.load_state_dict(checkpoint)
    return model.to(device)

model = load_original_model()
optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=weight_decay)

C:\tmp\ipykernel_65024\531252141.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load("result/vgg16/vgg16.pth")


In [96]:
# Using [Code11] as reference
n_epochs = 20
# Model has already been partially trained
model.train()

for epoch in range(n_epochs):
    # monitor training loss
    train_loss = 0.0
    train_loss1 = 0.0
    train_loss2 = 0.0
    
    ###################
    # train the model #
    ###################
    for data, target in trainloader:
        data, target = data.to(device), target.to(device) # loading to GPU
        # clear the gradients of all optimized variables
        optimizer.zero_grad()
        # forward pass: compute predicted outputs by passing inputs to the model
        output = model(data)
        # calculate the loss
        
        loss1 = criterion(output, target)
        loss2 = model.features[0].weight.abs().sum()

        gamma = 0.5 # multiply by this term now
        loss = loss1 + loss2 * gamma
        
        loss.backward()
        optimizer.step()
        train_loss1 += loss1.item()*data.size(0) # as loss is tensor, .item() needed to get the value
        train_loss2 += loss2.item()*data.size(0) # as loss is tensor, .item() needed to get the value
        train_loss  += loss.item()*data.size(0) # as loss is tensor, .item() needed to get the value
        
    # print training statistics 
    # calculate average loss over an epoch
    train_loss1 = train_loss1/len(trainloader.dataset)
    train_loss2 = train_loss2/len(trainloader.dataset)
    train_loss = train_loss/len(trainloader.dataset)

    print('Epoch: {} \tTraining Loss1: {:.6f}'.format(epoch+1, train_loss1))
    print('Epoch: {} \tTraining Loss2: {:.6f}'.format(epoch+1, train_loss2))
    print('Epoch: {} \tTraining Loss: {:.6f}'.format(epoch+1, train_loss))
  

Epoch: 1 	Training Loss1: 2.049947
Epoch: 1 	Training Loss2: 76.840484
Epoch: 1 	Training Loss: 40.470189
Epoch: 2 	Training Loss1: 1.822537
Epoch: 2 	Training Loss2: 76.986813
Epoch: 2 	Training Loss: 40.315944
Epoch: 3 	Training Loss1: 1.553580
Epoch: 3 	Training Loss2: 75.651949
Epoch: 3 	Training Loss: 39.379555
Epoch: 4 	Training Loss1: 1.342556
Epoch: 4 	Training Loss2: 76.061677
Epoch: 4 	Training Loss: 39.373395
Epoch: 5 	Training Loss1: 1.215858
Epoch: 5 	Training Loss2: 76.474025
Epoch: 5 	Training Loss: 39.452870
Epoch: 6 	Training Loss1: 1.149999
Epoch: 6 	Training Loss2: 76.077001
Epoch: 6 	Training Loss: 39.188499
Epoch: 7 	Training Loss1: 1.115353
Epoch: 7 	Training Loss2: 76.063178
Epoch: 7 	Training Loss: 39.146942
Epoch: 8 	Training Loss1: 1.056905
Epoch: 8 	Training Loss2: 77.705858
Epoch: 8 	Training Loss: 39.909834
Epoch: 9 	Training Loss1: 1.016394
Epoch: 9 	Training Loss2: 78.132130
Epoch: 9 	Training Loss: 40.082458
Epoch: 10 	Training Loss1: 0.995694
Epoch: 10 

In [97]:
# Evaluate the model with gamma = 0.1
criterion = nn.CrossEntropyLoss().cuda()

model.eval()
model.cuda()

prec = validate(testloader, model, criterion)

Test: [0/79]	Time 44.903 (44.903)	Loss 1.0969 (1.0969)	Prec 61.719% (61.719%)
 * Prec 61.660% 


In [98]:
# Get weight abs sum
print(model.features[0].weight.abs().sum())

tensor(77.8934, device='cuda:0', grad_fn=<SumBackward0>)


In [121]:
checkpoint = torch.load("result/vgg16/vgg16.pth")

# From hw4 solution
def load_original_model():
    model = VGG16()
    model.load_state_dict(checkpoint)
    return model.to(device)

model = load_original_model()
optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=weight_decay)

C:\tmp\ipykernel_65024\531252141.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load("result/vgg16/vgg16.pth")


In [122]:
# Evaluate the model with gamma = 0.1
criterion = nn.CrossEntropyLoss().cuda()

model.eval()
model.cuda()

prec = validate(testloader, model, criterion)


Test: [0/79]	Time 16.238 (16.238)	Loss 0.1696 (0.1696)	Prec 95.312% (95.312%)
 * Prec 92.170% 


In [123]:
# Using [Code11] as reference
n_epochs = 5
# Model has already been partially trained
model.train()

for epoch in range(n_epochs):
    # monitor training loss
    train_loss = 0.0
    train_loss1 = 0.0
    train_loss2 = 0.0
    
    ###################
    # train the model #
    ###################
    for data, target in trainloader:
        data, target = data.to(device), target.to(device) # loading to GPU
        # clear the gradients of all optimized variables
        optimizer.zero_grad()
        # forward pass: compute predicted outputs by passing inputs to the model
        output = model(data)
        # calculate the loss
        
        loss1 = criterion(output, target)
        loss2 = model.features[0].weight.abs().sum()

        gamma = 0.001 # multiply by this term now
        loss = loss1 + loss2 * gamma
        
        loss.backward()
        optimizer.step()
        train_loss1 += loss1.item()*data.size(0) # as loss is tensor, .item() needed to get the value
        train_loss2 += loss2.item()*data.size(0) # as loss is tensor, .item() needed to get the value
        train_loss  += loss.item()*data.size(0) # as loss is tensor, .item() needed to get the value
        
    # print training statistics 
    # calculate average loss over an epoch
    train_loss1 = train_loss1/len(trainloader.dataset)
    train_loss2 = train_loss2/len(trainloader.dataset)
    train_loss = train_loss/len(trainloader.dataset)

    print('Epoch: {} \tTraining Loss1: {:.6f}'.format(epoch+1, train_loss1))
    print('Epoch: {} \tTraining Loss2: {:.6f}'.format(epoch+1, train_loss2))
    print('Epoch: {} \tTraining Loss: {:.6f}'.format(epoch+1, train_loss))
  

Epoch: 1 	Training Loss1: 0.663431
Epoch: 1 	Training Loss2: 72.611271
Epoch: 1 	Training Loss: 0.736043
Epoch: 2 	Training Loss1: 0.525377
Epoch: 2 	Training Loss2: 54.448285
Epoch: 2 	Training Loss: 0.579825
Epoch: 3 	Training Loss1: 0.505642
Epoch: 3 	Training Loss2: 49.175260
Epoch: 3 	Training Loss: 0.554817
Epoch: 4 	Training Loss1: 0.483672
Epoch: 4 	Training Loss2: 45.140501
Epoch: 4 	Training Loss: 0.528812
Epoch: 5 	Training Loss1: 0.480049
Epoch: 5 	Training Loss2: 43.861701
Epoch: 5 	Training Loss: 0.523911


In [124]:
# Evaluate the model with gamma = 0.001
criterion = nn.CrossEntropyLoss().cuda()

model.eval()
model.cuda()

prec = validate(testloader, model, criterion)

Test: [0/79]	Time 16.347 (16.347)	Loss 0.7858 (0.7858)	Prec 76.562% (76.562%)
 * Prec 75.550% 


In [125]:
# Get weight abs sum
print(model.features[0].weight.abs().sum())

tensor(42.9138, device='cuda:0', grad_fn=<SumBackward0>)
